# Việt Nam đứng ở đâu trong ASEAN về phát thải CO2? Reshape long ↔ wide và tăng trưởng theo nước

Bộ `owid-co2` (Our World in Data, CC BY 4.0, ghim commit `382ee6c`): `owid-co2-data.csv`, 50 411 dòng, mỗi dòng một
cặp `country`, `year` (1750–2024), 79 cột. Cột dùng trong bài: `country`, `year`, `iso_code`, `co2` (triệu tấn CO2
từ nhiên liệu hoá thạch và công nghiệp, **không** tính thay đổi sử dụng đất), `co2_per_capita` (tấn/người).

**Bối cảnh.** Ban phát triển bền vững của một tập đoàn chuẩn bị báo cáo ESG cho thị trường ASEAN và hỏi: *"Việt Nam
phát thải nhiều cỡ nào so với các nước ASEAN khác, và đang tăng nhanh hay chậm?"*. Họ cần một bảng 11 nước ASEAN
(gồm Đông Timor, thành viên thứ 11 từ 10/2025) với giá trị 2014, 2024, tăng trưởng 10 năm và CAGR, cùng chuỗi tăng
trưởng từng năm của Việt Nam.

Bảng OWID là **long** nhưng trộn dòng quốc gia với dòng tổng (`World`, châu lục, nhóm thu nhập, hàng không quốc tế...):
các dòng tổng có `iso_code` rỗng.

**Cách làm bài**
- Cell chấm điểm chỉ định nghĩa hàm. Điền vào chỗ `# TODO`; hàm không được sửa DataFrame đầu vào.
- Cell gắn tag `explore` (dòng đầu `# explore: ...`) đọc dữ liệu thật và vẽ; khi chấm chúng bị bỏ qua.
- Chấm: `uv run pytest tracks/data-analyst/modules/04-python-for-analysts/exercises/basic/01-co2-reshape-growth -q`

In [ ]:
# explore: tìm gốc repo, tải bộ dữ liệu nếu chưa có, đọc CSV
import sys
from pathlib import Path

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
sys.path.insert(0, str(ROOT))
from data.download_dataset import ensure
import pandas as pd

co2 = pd.read_csv(ensure("owid-co2") / "owid-co2-data.csv")
print(co2.shape, co2.year.min(), "→", co2.year.max())
print("dòng iso_code rỗng:", co2.iso_code.isna().sum(), "| ví dụ:", co2.loc[co2.iso_code.isna(), "country"].unique()[:6].tolist())

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

ASEAN = ["Brunei", "Cambodia", "East Timor", "Indonesia", "Laos", "Malaysia", "Myanmar", "Philippines",
         "Singapore", "Thailand", "Vietnam"]

## 1. Bỏ dòng tổng

`countries_only(df) -> pd.DataFrame`: chỉ giữ dòng có `iso_code` khác rỗng (quốc gia và vùng lãnh thổ), giữ nguyên
thứ tự dòng, index và mọi cột.

In [ ]:
def countries_only(df: pd.DataFrame) -> pd.DataFrame:
    """Dòng có iso_code khác rỗng; không sửa df."""
    return df[df["iso_code"].notna()].copy()

In [ ]:
# explore: cộng cả cột co2 năm 2024 trước và sau khi bỏ dòng tổng, so với dòng World
y24 = co2[co2.year == 2024]
world = y24.loc[y24.country == "World", "co2"].item()
nations = countries_only(co2)
print(f"cộng mọi dòng: {y24.co2.sum():,.0f} Mt ({y24.co2.sum() / world:.1f} lần World) | World: {world:,.0f} Mt")
print(f"cộng {nations.country.nunique()} quốc gia: {nations.loc[nations.year == 2024, 'co2'].sum():,.0f} Mt")
print(y24[y24.country.str.startswith("International")][["country", "co2"]])

## 2. Long → wide → long

- `to_wide(df, value, countries=None, years=None) -> pd.DataFrame`: mỗi nước một dòng, mỗi năm một cột, ô là cột
  `value`. Index tên `country`, tên trục cột là `year`. Dùng `pivot` (không phải `pivot_table`): khoá trùng phải báo
  lỗi. Nếu truyền `countries` thì dòng theo đúng thứ tự danh sách đó; nếu truyền `years` thì cột theo đúng danh sách
  đó. Nước hoặc năm không có dữ liệu thì ô là `NaN`.
- `to_long(wide, value) -> pd.DataFrame`: ngược lại, ba cột `country`, `year` (kiểu `int`), `value`; bỏ ô `NaN`; sắp
  theo `country` rồi `year`; index `0..n-1`.

In [ ]:
def to_wide(df: pd.DataFrame, value: str, countries: list[str] | None = None,
            years: list[int] | None = None) -> pd.DataFrame:
    """Index country, cột là year, ô là df[value]."""
    wide = df.pivot(index="country", columns="year", values=value)
    if countries is not None:
        wide = wide.reindex(countries)
    if years is not None:
        wide = wide.reindex(columns=years)
    wide.index.name = "country"
    wide.columns.name = "year"
    return wide


def to_long(wide: pd.DataFrame, value: str) -> pd.DataFrame:
    """Cột country, year (int), value; bỏ NaN; sắp theo country, year."""
    long = (wide.rename_axis(index="country", columns="year")
                .reset_index()
                .melt(id_vars="country", var_name="year", value_name=value)
                .dropna(subset=[value]))
    long["year"] = long["year"].astype(int)
    return long.sort_values(["country", "year"]).reset_index(drop=True)

In [ ]:
# explore: bảng wide ASEAN 2014-2024 và kiểm vòng long -> wide -> long
asean_wide = to_wide(nations, "co2", countries=ASEAN, years=list(range(2014, 2025)))
print(asean_wide.round(1).to_string())
back = to_long(asean_wide, "co2")
orig = nations[nations.country.isin(ASEAN) & nations.year.between(2014, 2024)][["country", "year", "co2"]]
print("vòng long -> wide -> long giữ nguyên dữ liệu:",
      back.equals(orig.sort_values(["country", "year"]).reset_index(drop=True)))

## 3. Tăng trưởng giữa hai năm và CAGR

`growth_table(wide, start, end) -> pd.DataFrame`: từ bảng wide (cột là năm), mỗi nước một dòng với bốn cột theo thứ
tự `start`, `end` (giá trị hai năm), `pct_change` (`end / start - 1`) và `cagr`:

$$
\text{CAGR} = \left(\frac{\text{end}}{\text{start}}\right)^{1/(\text{năm cuối} - \text{năm đầu})} - 1
$$

Bỏ nước thiếu giá trị một trong hai năm hoặc có `start <= 0` (tăng trưởng không xác định). Sắp theo `cagr` giảm dần.

In [ ]:
def growth_table(wide: pd.DataFrame, start: int, end: int) -> pd.DataFrame:
    """Cột start, end, pct_change, cagr; sắp theo cagr giảm dần."""
    out = pd.DataFrame({"start": wide[start], "end": wide[end]}).dropna()
    out = out[out["start"] > 0]
    out["pct_change"] = out["end"] / out["start"] - 1
    out["cagr"] = (out["end"] / out["start"]) ** (1 / (end - start)) - 1
    return out.sort_values("cagr", ascending=False)

In [ ]:
# explore: tăng trưởng 10 năm của ASEAN, tổng và đầu người
g_total = growth_table(asean_wide, 2014, 2024)
print(g_total.round(3).to_string())
pc_wide = to_wide(nations, "co2_per_capita", countries=ASEAN, years=[2014, 2024])
print(growth_table(pc_wide, 2014, 2024).round(3).to_string())
print(f"Việt Nam chiếm {asean_wide.loc['Vietnam', 2024] / asean_wide[2024].sum():.1%} CO2 của ASEAN năm 2024")
print("thế giới, tấn/người 2024:", co2.loc[(co2.country == "World") & (co2.year == 2024), "co2_per_capita"].item())

In [ ]:
# explore: dot plot 2014 -> 2024, mỗi nước một hàng, sắp theo giá trị 2024
order = asean_wide[2024].sort_values().index
fig, ax = plt.subplots(figsize=(7, 4))
ax.hlines(order, asean_wide.loc[order, 2014], asean_wide.loc[order, 2024], color="#c0c7d0")
ax.scatter(asean_wide.loc[order, 2014], order, color="#9aa5b1", label="2014", zorder=3)
ax.scatter(asean_wide.loc[order, 2024], order, color="#4c78a8", label="2024", zorder=3)
ax.set_xscale("log")
ax.set_xlabel("triệu tấn CO2 (thang log)")
ax.legend(frameon=False)
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("ASEAN: Việt Nam đứng thứ hai về tổng phát thải, sau Indonesia", loc="left")
plt.show()

## 4. Tăng trưởng so với **năm** trước, không phải dòng trước

`yoy(long, value) -> pd.DataFrame`: nhận bảng long (`country`, `year`, `value`, có thể thêm cột khác), trả bản sao có
thêm cột `yoy` = giá trị năm đó / giá trị **năm liền trước của cùng nước** − 1. Nếu nước đó không có dòng năm liền
trước (hoặc giá trị năm trước rỗng) thì `yoy` là `NaN`. Giữ nguyên thứ tự dòng và index của đầu vào.

`groupby("country")[value].pct_change()` **không** đúng yêu cầu: nó so với dòng trước, kể cả khi dòng trước cách
50 năm.

In [ ]:
def yoy(long: pd.DataFrame, value: str) -> pd.DataFrame:
    """Thêm cột yoy so với năm liền trước của cùng nước; thiếu năm trước thì NaN."""
    prev = (long[["country", "year", value]]
            .assign(year=long["year"] + 1)
            .rename(columns={value: "_prev"}))
    merged = long[["country", "year"]].merge(prev, on=["country", "year"], how="left", validate="many_to_one")
    out = long.copy()
    out["yoy"] = (long[value].to_numpy() / merged["_prev"].to_numpy()) - 1
    return out

In [ ]:
# explore: Việt Nam từng năm, và một nước có lỗ hổng dữ liệu (Eritrea: 1944 rồi nhảy sang 1994)
vn = yoy(nations.loc[nations.country == "Vietnam", ["country", "year", "co2"]], "co2")
print(vn[vn.year >= 2015].round(4).to_string(index=False))
er = nations.loc[nations.country == "Eritrea", ["country", "year", "co2"]].dropna()
er = er.assign(pct_change_dong_truoc=er.groupby("country")["co2"].pct_change())
print(yoy(er, "co2").query("1943 <= year <= 1995").round(3).to_string(index=False))

In [ ]:
# explore: tăng trưởng từng năm của Việt Nam 2015-2024 (cột), năm âm tô màu khác
vn10 = vn[vn.year >= 2015]
fig, ax = plt.subplots(figsize=(7, 3))
ax.bar(vn10.year, vn10.yoy * 100, color=np.where(vn10.yoy >= 0, "#4c78a8", "#e45756"))
ax.axhline(0, color="black", lw=0.8)
ax.set_ylabel("% so với năm trước")
ax.spines[["top", "right"]].set_visible(False)
ax.set_title("CO2 Việt Nam: năm nào tăng vọt?", loc="left")
plt.show()

## 5. Đọc kết quả, rồi làm trắc nghiệm

Viết cho ban phát triển bền vững ba câu:

1. Việt Nam xếp thứ mấy trong ASEAN về tổng phát thải 2024, chiếm bao nhiêu phần ASEAN, và đầu người so với trung
   bình thế giới thế nào?
2. Tăng trưởng 10 năm (CAGR) của Việt Nam so với các nước ASEAN khác ra sao? Nước nào tăng nhanh nhất, và vì sao một
   tỉ lệ tăng rất lớn có thể không đáng lo bằng một tỉ lệ nhỏ hơn?
3. Năm nào CO2 Việt Nam tăng vọt nhất trong 2015–2024? Bạn sẽ hỏi thêm dữ liệu gì để giải thích (gợi ý: cột
   `coal_co2`)?

Xong thì mở app, làm phần **trắc nghiệm** của bài.